## Using STED-FM model in PhenoMe pipeline

### 0. Setup

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from stedfm import get_pretrained_model_v2
from phenome.utils import ModelWrapper
from phenome import PhenoMe, load_dinov2_model

# Initialize PhenoMe object
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

#### Choose dataset and model to use

**NAS:** 224x224 crops of STED images of PSD-95 under 4 experimental conditions: 0MgGlyBic, 48hTTX, Block, and GluGly.

**Optim:** 224x224 crops of STED images of 4 different proteins: CaMKII, PSD-95, Actin and Tubulin.

**Zooniverse:** Crops of individual synaptic protein clusters. This is the only dataset with masks appropriate for extracting properties.


In [ ]:
# Choose one of 3 datasets from ("NAS", "Optim", "Zooniverse")
DATASET_NAME = "NAS"
#DATASET_NAME = "Optim"
#DATASET_NAME = "Zooniverse"

# Choose one of 2 models from ("STED-FM", "DinoV2")
MODEL = "STED-FM"
#MODEL = "DinoV2"

#### Download and transform data

`DATASETS_PATH` can be changed to wherever the user wans the datasets to be downloaded to.

In [ ]:
# Decide where to save datasets
DATASETS_PATH = "stedfm_datasets"
os.makedirs(DATASETS_PATH, exist_ok=True)

DATA_DIR = os.path.join(DATASETS_PATH, f"{DATASET_NAME}_dataset")
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
from etl import download_dataset, transform_dataset

# Download and transform dataset to fit PhenoMe's required structure
download_dataset(DATASET_NAME, DATA_DIR)
transform_dataset(DATASET_NAME, DATA_DIR, DATA_DIR)

### 1. Wrap the STED-FM pretrained model

In [ ]:
class STEDFMWrapper(ModelWrapper):
    def __init__(self, device):
        # Load the pretrained STED-FM model
        model, cfg = get_pretrained_model_v2(
            name = "mae-lightning-small",
            weights = "MAE_SMALL_STED", 
            in_channels=1,
            as_classifier=True,
        )

        # Initialize the parent class which moves the model to the device and sets it to eval()
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        out = self.model.forward_features(tensor)
        return out

if MODEL == "STED-FM":
    wrapper = STEDFMWrapper(device=device)
elif MODEL == "DinoV2":
    wrapper = load_dinov2_model(model_name="dinov2_vits14_reg", device=device)

### 2. Use custom transforms

The default PhenoMe transforms apply standardisation based on ImageNet values, which isn't appropriate for the STED-FM model.

In [ ]:
import torchvision.transforms as T

TARGET_SIZE = 224

if MODEL == "STED-FM":
    # Transforms for STED-FM model
    custom_transforms = T.Compose([
        T.ToTensor(),
        T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
    ])
elif MODEL == "DinoV2":
    # Transforms for DinoV2 model
    custom_transforms = T.Compose([
        T.ToTensor(),
        T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])

### 3. Load data (find_files)

We assume CSV-based metadata. The STED datasets were already normalized. If raw data is used, it should be normalized based on percentile values from **entire images**, not by crop.

In [ ]:

from phenome import make_dataframe_metadata_fn
import pandas as pd

images_dir = os.path.join(DATA_DIR, "images")
masks_dir = os.path.join(DATA_DIR, "masks") # Optional
metadata_dir = os.path.join(DATA_DIR, "metadata.csv")

metadata_df = pd.read_csv(metadata_dir)
metadata_fn = make_dataframe_metadata_fn(metadata_df)
file_df = pheno.find_files(images_dir, mask_dir=masks_dir, metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")

### 4. Process images

In [ ]:
# Base path to outputs: checkpoints and plots
OUTPUT_PATH = "output"
os.makedirs(OUTPUT_PATH, exist_ok=True)

CHECKPOINTS_PATH = os.path.join(OUTPUT_PATH, "checkpoints")
os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

In [ ]:
checkpoint_path = os.path.join(CHECKPOINTS_PATH, DATASET_NAME + f"_{MODEL}" + "_checkpoint.hdf5")

if MODEL == "STED-FM":
    pheno.process_images(wrapper, 
                        force_rgb=False,  # STED-FM model expects a single channel
                        checkpoint_path=checkpoint_path,
                        custom_transformations=custom_transforms,
                        )
elif MODEL == "DinoV2":
    pheno.process_images(wrapper, 
                    force_rgb=True,
                    channel_mode='combined',
                    checkpoint_path=checkpoint_path,
                    custom_transformations=custom_transforms,
                    )
pheno.compute_properties(property_preset="complete", checkpoint_path=checkpoint_path)

### 5. Explore results

In [ ]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()

### Generate plots

In [ ]:
PLOTS_PATH = os.path.join(OUTPUT_PATH, "plots")
os.makedirs(PLOTS_PATH, exist_ok=True)

if DATASET_NAME == "NAS":
    color_by = "condition"
    fig=pheno.plot_tsne(color_by=color_by, return_fig=True, render_mode="svg", exclude={"condition":"Block"})
    fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_noblock_scatterplot.svg"))
elif DATASET_NAME == "Optim":
    color_by = "label"
elif DATASET_NAME == "Zooniverse":
    color_by = "protein"

fig=pheno.plot_tsne(color_by=color_by, return_fig=True, render_mode="svg")
fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_scatterplot.svg"))